# Προ-Επεξεργασία των Εικόνων

## Φόρτωση των δεδομένων

In [5]:
import os
import tensorflow as tf
import csv
import numpy as np
from pathlib import Path
import kagglehub
from PIL import Image
import matplotlib.pyplot as plt
from kagglehub import KaggleDatasetAdapter

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

path = Path(kagglehub.dataset_download("marquis03/fruits-100"))
filename_names = path / "classname.txt"
filename_train = path / "train.csv"
filename_test = path / "test.csv"
# filename_vali = path / "val.csv"


# load data
def take_fruit_names(filename):
    fruit_names = {}
    fruit_id  = 0
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(',')
            fruit_name = parts[0]
            fruit_names[fruit_id] = fruit_name
            fruit_id += 1            
    return fruit_names


def load_data_fruit_paths(path):
   image_paths = []
   with open(path, 'r', newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            image_paths.append(row[0])
   return image_paths

def load_data_fruit_category(path):
   image_category = []
   with open(path, 'r', newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            image_category.append(row[1])
   return image_category

fruit_names = take_fruit_names(filename_names)
train_image_paths = load_data_fruit_paths(filename_train)
train_image_category = load_data_fruit_category(filename_train)

test_image_paths = load_data_fruit_paths(filename_test)
print("OK")


OK


## Αλλαγή Μεγέθους και Κανονικοποίηση των Εικόνων

In [9]:
import tensorflow as tf
import csv
import numpy as np
from pathlib import Path
import kagglehub
from PIL import Image
from kagglehub import KaggleDatasetAdapter

def take_picture(image_path, image_size=(150, 150)):
    img = Image.open(image_path).convert('RGB')  # άνοιγμα σε RGB
    img = img.resize(image_size)  # αλλαγή μεγέθους (προαιρετικά)
    return np.array(img)               
    
def pictures_to_0_1(images):
    images = np.asarray(images, dtype=np.float32)
    images /= 255.0
    return images

image_fruit_train_image  = []
image_fruit_train_image_labels = []

for img_path, id in zip(train_image_paths, train_image_category):
    id = int(id)  # βεβαιωνόμαστε ότι είναι int
    full_path = path / img_path  # img_path από το CSV
    try:
        img_array = take_picture(full_path, image_size=(150,150))
        # Κανονικοποίηση της εικόνας
        img_array = pictures_to_0_1(img_array)
        image_fruit_train_image.append(img_array)
        image_fruit_train_image_labels.append(id)
    except Exception as e:
        print(f"Error loading image {full_path}: {e}")
print("Φόρτωση ολοκληρώθηκε:", len(image_fruit_train_image), "εικόνες στο train")
print("Φόρτωση ολοκληρώθηκε:", len(image_fruit_train_image_labels), "labels στο train")

#Δεδομένα του test
image_fruit_test_image  = []
for img_path in test_image_paths:
    full_path = path / img_path  # img_path από το CSV
    try:
        img_array = take_picture(full_path, image_size=(150,150))
        img_array = pictures_to_0_1(img_array)
        image_fruit_test_image.append(img_array)
    except Exception as e:
        print(f"Error loading image {full_path}: {e}")

print("Φόρτωση ολοκληρώθηκε:", len(image_fruit_test_image), "εικόνες στο test")

Φόρτωση ολοκληρώθηκε: 40000 εικόνες στο train
Φόρτωση ολοκληρώθηκε: 40000 labels στο train
Φόρτωση ολοκληρώθηκε: 5000 εικόνες στο test


## Αποθήκευση των Εικόνων στο Δίσκο

In [10]:
import numpy as np

# Μέγεθος εικόνων
num_test_images = len(image_fruit_test_image)
num_train_images = len(image_fruit_train_image)
num_train_labels = len(image_fruit_train_image_labels)
height, width, channels = image_fruit_train_image[0].shape

# Όνομα αρχείου για memmap
filename_test_image = "fruit_test_images.npy"
filename_train_image = "fruit_train_images.npy"
filename_train_image_labels = "fruit_train_images_labels.npy"

# Δημιουργία memmap στο δίσκο
fp_test = np.memmap(filename_test_image, dtype='float32', mode='w+', shape=(num_test_images, height, width, channels))
fp_train = np.memmap(filename_train_image, dtype='float32', mode='w+', shape=(num_train_images, height, width, channels))
fp_train_labels = np.memmap(filename_train_image_labels, dtype='float32', mode='w+',shape=(num_train_labels,))

# Γράφουμε τις test εικόνες στο memmap τμηματικά
for idx, img_array in enumerate(image_fruit_test_image):
    fp_test[idx] = img_array

# Γράφουμε τις train εικόνες στο memmap τμηματικά
for idx, img_array in enumerate(image_fruit_train_image):
    fp_train[idx] = img_array

# Γράφουμε τις train εικόνες στο memmap τμηματικά
for idx, img_array in enumerate(image_fruit_train_image_labels):
    fp_train_labels[idx] = img_array
    
# Σιγουρευόμαστε ότι όλα γράφτηκαν
fp_test.flush()
fp_train.flush()
fp_train_labels.flush()

print("Test images αποθηκεύτηκαν στο memmap:", filename_test_image)
print("Train images αποθηκεύτηκαν στο memmap:", filename_train_image)
print("Τα train labels αποθηκεύτηκαν στο memmap:", num_train_labels)


Test images αποθηκεύτηκαν στο memmap: fruit_test_images.npy
Train images αποθηκεύτηκαν στο memmap: fruit_train_images.npy
Τα train labels αποθηκεύτηκαν στο memmap: 40000
